# Choose between `if`/`elif` chains, dict dispatch, and `match`/`case`

Three legitimate ways to branch on a value: `if`/`elif`, a lookup table (dict), and `match`/`case`. Each has a sweet spot. This recipe walks through the same problem in all three styles and lays out the criteria for choosing.

## The example: routing HTTP responses

Imagine you receive an HTTP response — a dict with a `status` field and a body — and need to decide what to do next.

In [ ]:
# Sample responses we'll route
ok_response       = {"status": 200, "body": {"data": [1, 2, 3]}}
created_response  = {"status": 201, "body": {"id": 42}}
not_found         = {"status": 404, "body": {"error": "missing"}}
server_error      = {"status": 500, "body": {"error": "internal"}}
unknown           = {"status": 999, "body": {}}

## Approach 1: `if`/`elif` chain

The most direct version. Every branch is a free-form block.

In [ ]:
def route_if(response):
    status = response["status"]
    if status == 200:
        return f"OK — got {len(response['body']['data'])} items"
    elif status == 201:
        return f"Created with id {response['body']['id']}"
    elif status == 404:
        return "Not found — fall back to cache"
    elif status == 500:
        return "Server error — back off and retry"
    else:
        return f"Unexpected status {status}"

for r in [ok_response, created_response, not_found, server_error, unknown]:
    print(route_if(r))

**Strengths:** No setup. Every branch can do anything — call functions, raise exceptions, log, return early. The reader sees exactly what runs.

**Weaknesses:** Doesn't scale to dozens of branches. Repeats the `status == ...` boilerplate. Adding a new case requires editing the chain (and possibly its order).

## Approach 2: dict dispatch

When each branch is "look up by key, call a handler", a dict is often cleaner. Each handler is a small function — the routing logic disappears into the dict.

In [ ]:
def handle_ok(response):       return f"OK — got {len(response['body']['data'])} items"
def handle_created(response):  return f"Created with id {response['body']['id']}"
def handle_not_found(_):       return "Not found — fall back to cache"
def handle_server_error(_):    return "Server error — back off and retry"
def handle_unknown(response):  return f"Unexpected status {response['status']}"

ROUTES = {
    200: handle_ok,
    201: handle_created,
    404: handle_not_found,
    500: handle_server_error,
}

def route_dict(response):
    handler = ROUTES.get(response["status"], handle_unknown)
    return handler(response)


for r in [ok_response, created_response, not_found, server_error, unknown]:
    print(route_dict(r))

**Strengths:** Adding a new case is one line in the dict. Handlers can be tested in isolation. The dict can be extended at runtime — useful for plugin systems. Easy to introspect ("what does the system handle?" = `ROUTES.keys()`).

**Weaknesses:** Splits the logic across multiple definitions. Each branch is a function, which is overkill for a one-line response. No destructuring — the handler still has to dig into the response itself.

## Approach 3: `match`/`case`

`match` does shape-matching and destructuring in one step. For data with structure, this is its sweet spot.

In [ ]:
def route_match(response):
    match response:
        case {"status": 200, "body": {"data": data}}:
            return f"OK — got {len(data)} items"
        case {"status": 201, "body": {"id": id}}:
            return f"Created with id {id}"
        case {"status": 404}:
            return "Not found — fall back to cache"
        case {"status": 500}:
            return "Server error — back off and retry"
        case {"status": status}:
            return f"Unexpected status {status}"


for r in [ok_response, created_response, not_found, server_error, unknown]:
    print(route_match(r))

**Strengths:** The branch is the shape *and* the destructuring. No `response["body"]["data"]` boilerplate. New cases involve adding a new pattern, not extending a chain.

**Weaknesses:** Patterns syntax has its own gotchas (the capture-vs-compare trap, `case CONST:` not doing what you'd expect). Requires Python 3.10+. Less familiar to readers — a newcomer to `match` may need to look things up.

## Decision criteria

| Use this... | When... |
|-------------|---------|
| **`if`/`elif`** | Few branches; each branch does free-form work; conditions involve more than equality (e.g. `if score > 0.8`). |
| **Dict dispatch** | Many branches keyed on a single value; each branch is a callable; you want runtime-extensible routing or want to inspect the supported set. |
| **`match`/`case`** | Branching depends on **shape** as well as value; you want to destructure dict/dataclass/tuple input; the set of cases is closed and enumerable. |

In practice, the choice depends on what's likely to change. If branches grow in count but stay shaped the same, dict dispatch wins. If new branches need new shapes (different fields, different types), `match` wins. If branches are heterogeneous one-offs, `if`/`elif` is fine.

## A common pattern: combine them

These approaches aren't exclusive. A real handler often uses two together — `match` to dispatch on shape, then a dict inside one of the branches for fine-grained routing:

In [ ]:
ERROR_HANDLERS = {
    "rate_limit":   lambda r: f"Slow down: {r['retry_after']}s",
    "auth_failed":  lambda _: "Reauthenticate",
    "server_busy":  lambda _: "Back off and retry",
}

def route_combined(response):
    match response:
        case {"status": 200, "body": {"data": data}}:
            return f"OK — got {len(data)} items"
        case {"status": 4 | 5, "body": {"error": kind}} as r if kind in ERROR_HANDLERS:
            # actually unreachable — `4 | 5` matches the literal int 4 or 5,
            # not "any 4xx". Use a guard instead:
            return ERROR_HANDLERS[kind](r["body"])
        case {"status": status, "body": {"error": kind}} if 400 <= status < 600:
            handler = ERROR_HANDLERS.get(kind, lambda _: f"Error: {kind}")
            return handler(response.get("body", {}))
        case {"status": status}:
            return f"Unhandled status {status}"


# Demo
print(route_combined({"status": 200, "body": {"data": [1, 2]}}))
print(route_combined({"status": 429, "body": {"error": "rate_limit", "retry_after": 30}}))
print(route_combined({"status": 401, "body": {"error": "auth_failed"}}))

(The first `case` shows what *doesn't* work — `4 | 5` is a literal-OR pattern, not a "starts with 4 or 5" expression. The next case uses an `if` guard for the range check, which does work.)

The pattern: use `match` to peel off the structural shape, and reach for a dict when you've narrowed down to a single dimension with many possible values.

## Related reading

- [`match`/`case` syntax](../reference/match-case-syntax.md) — every pattern type at a glance.
- [Structural pattern matching in context](../concepts/structural-pattern-matching-in-context.md) — when `match` earns its place and when it doesn't.
- [Use guard clauses to flatten nested conditions](use-guard-clauses.ipynb) — the refactor that often comes before any of the dispatch styles.